# 7. Organization AI Maturity Report

Scores **all repositories** in an organization's output directory using the AIME maturity framework (L1–L4),
then aggregates into an executive-level dashboard with:

1. **Maturity Heatmap** — treemap of all repos colored by level
2. **Temporal Evolution** — monthly % of repos at each maturity level over time
3. **Top Repositories** — ranked by maturity level and confidence
4. **Top Contributors** — ranked by AI artifact commits (anonymized)
5. **Tool Adoption** — which AI tools are used across the org
6. **Category Gap Analysis** — underrepresented maturity categories
7. **Artifact Density** — distribution of artifact counts per repo
8. **Risk Flags** — repos with temporal health concerns

**Model**: `nomic-ai/nomic-embed-text-v1.5` (768 dims, loaded once for all repos)

In [ ]:
# IMPORTANT: Set OpenMP environment variables BEFORE any other imports
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_MAX_ACTIVE_LEVELS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import json
import pickle
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.embedding_generator import load_embedding_model, DEFAULT_MODEL
from src.maturity_scorer import (
    CATEGORY_NAMES,
    CATEGORY_TO_LEVEL,
    MATURITY_LABELS,
    MaturityLevel,
    score_from_output_dir,
)
from src.temporal_health import (
    analyze_temporal_health,
    load_timeseries,
    GROUNDING_CATEGORIES,
)
from src.report_generator import fig_to_base64, _import_weasyprint, _markdown_to_html

print(f"Model: {DEFAULT_MODEL}")
print(f"Categories ({len(CATEGORY_NAMES)}): {CATEGORY_NAMES}")

## Configuration

In [ ]:
PROJECT_ROOT = Path("..").resolve()

ORG_NAME = "example-run"
OUTPUT_DIR = (PROJECT_ROOT.parent / "collector" / "output" / ORG_NAME).resolve()

ARTIFACTS_DIR = str(PROJECT_ROOT / "Artifacts")
EXPORT_DIR = PROJECT_ROOT / "data"
EXPORT_DIR.mkdir(exist_ok=True)

# Level colors
LEVEL_COLORS = {
    1: "#6b7280",  # gray
    2: "#3b82f6",  # blue
    3: "#f97316",  # orange
    4: "#22c55e",  # green
}
LEVEL_LABELS = {
    1: "L1 Ad Hoc",
    2: "L2 Grounded",
    3: "L3 Agent-Augmented",
    4: "L4 Orchestration",
}

print(f"Organization: {ORG_NAME}")
print(f"Output dir: {OUTPUT_DIR}")

## Phase 1: Score All Repositories

In [ ]:
# Discover all repos
repos = sorted([
    item.name for item in OUTPUT_DIR.iterdir()
    if item.is_dir() and not item.name.startswith('.')
    and list(item.glob('*_file_artifacts.csv'))
])
print(f"Found {len(repos)} repositories in {OUTPUT_DIR.name}/")

In [ ]:
# Load model once
model = load_embedding_model(DEFAULT_MODEL)

# Score every repo
repo_scores = {}   # repo_name -> MaturityScore
repo_temporal = {} # repo_name -> TemporalHealth
failed_repos = []

for repo in tqdm(repos, desc="Scoring repos"):
    try:
        score = score_from_output_dir(
            str(OUTPUT_DIR), repo, model,
            artifacts_dir=ARTIFACTS_DIR,
        )
        repo_scores[repo] = score

        # Temporal health (if file_classifications available)
        if score.file_classifications is not None and not score.file_classifications.empty:
            try:
                th = analyze_temporal_health(
                    repo_dir=str(OUTPUT_DIR / repo),
                    file_classifications=score.file_classifications,
                    repo_name=repo,
                )
                if th.has_timeseries:
                    repo_temporal[repo] = th
            except Exception:
                pass
    except Exception as e:
        failed_repos.append((repo, str(e)))

print(f"\nScored: {len(repo_scores)} repos")
print(f"Temporal data: {len(repo_temporal)} repos")
print(f"Failed: {len(failed_repos)} repos")
if failed_repos:
    for name, err in failed_repos[:5]:
        print(f"  {name}: {err}")

## Phase 2: Aggregate Org Metrics

In [ ]:
# ── Filter out false positives ──
# Repos where ALL artifacts have tool_name='unknown' are not genuine AI adopters.
# They only contain generic files (README.md, etc.) that got misclassified by embeddings.
# We override their level to L1 (Ad Hoc).

GENUINE_DISCOVERY_STEPS = {'tool_standard', 'shared_in_tool_folder'}

overridden_to_l1 = 0
for repo, score in repo_scores.items():
    fc_df = score.file_classifications
    if fc_df is None or fc_df.empty:
        continue

    has_known_tool = (fc_df['tool_name'] != 'unknown').any()
    has_standard_discovery = fc_df['discovery_step'].isin(GENUINE_DISCOVERY_STEPS).any()

    if not has_known_tool and not has_standard_discovery:
        # Override to L1 — this repo has no genuine AI tool artifacts
        score.overall_level = 1
        score.overall_label = "Ad Hoc"
        score.confidence = 1.0
        overridden_to_l1 += 1

print(f"Repos overridden to L1 (no genuine AI artifacts): {overridden_to_l1}")

# Build batch summary DataFrame
batch_rows = []
for repo, score in repo_scores.items():
    batch_rows.append({
        'repo': repo,
        'level': score.overall_level,
        'label': score.overall_label,
        'confidence': score.confidence,
        'artifacts': score.artifact_count,
        'tools': ', '.join(score.tools_detected),
        'l2_primary': score.level_evidence.get(2, {}).get('primary', 0),
        'l3_primary': score.level_evidence.get(3, {}).get('primary', 0),
        'l4_primary': score.level_evidence.get(4, {}).get('primary', 0),
    })
batch_df = pd.DataFrame(batch_rows)

# Level distribution
level_dist = batch_df['level'].value_counts().sort_index()
total_repos = len(batch_df)
scored_with_artifacts = (batch_df['level'] > 1).sum()

print(f"\n{'='*60}")
print(f"ORGANIZATION MATURITY OVERVIEW: {ORG_NAME.upper()}")
print(f"{'='*60}")
print(f"Total repositories: {total_repos}")
print(f"With AI artifacts (L2+): {scored_with_artifacts} ({scored_with_artifacts/total_repos*100:.0f}%)")
print()
print("Level Distribution:")
for lvl in [1, 2, 3, 4]:
    count = level_dist.get(lvl, 0)
    pct = count / total_repos * 100
    bar = '█' * int(pct / 2)
    print(f"  {LEVEL_LABELS[lvl]:25s} {count:4d} ({pct:5.1f}%) {bar}")

In [ ]:
# Compute when each repo first reached each maturity level
# by joining timeseries with file classifications
# Only for genuine L2+ repos (not overridden to L1)

level_reached = {}  # repo -> {2: Timestamp, 3: Timestamp, 4: Timestamp}
genuine_repos = {repo for repo, score in repo_scores.items() if score.overall_level >= 2}

for repo in tqdm(genuine_repos, desc="Computing level dates"):
    score = repo_scores[repo]
    fc_df = score.file_classifications
    if fc_df is None or fc_df.empty:
        continue

    ts_df = load_timeseries(str(OUTPUT_DIR / repo), repo)
    if ts_df is None or ts_df.empty:
        continue

    # Join timeseries with classifications on artifact_path
    fc_cats = fc_df[['artifact_path', 'assigned_category', 'assigned_maturity_level']].copy()
    merged = ts_df.merge(fc_cats, on='artifact_path', how='inner')

    if merged.empty:
        continue

    dates = {}
    for lvl in [2, 3, 4]:
        lvl_rows = merged[merged['assigned_maturity_level'] == lvl]
        if not lvl_rows.empty:
            # Prefer 'created' action, fall back to any
            created = lvl_rows[lvl_rows['action'] == 'created']
            src = created if not created.empty else lvl_rows
            dates[lvl] = src['commit_date'].min()

    if dates:
        level_reached[repo] = dates

print(f"Genuine L2+ repos: {len(genuine_repos)}")
print(f"Repos with level-reached dates: {len(level_reached)}")

## Chart 1: Maturity Heatmap

In [ ]:
# Maturity heatmap: waffle-style grid of all repos
# Each square = 1 repo, colored by maturity level, sorted by level desc

sorted_repos = batch_df.sort_values(
    ['level', 'artifacts'], ascending=[False, False]
).reset_index(drop=True)

# Grid layout: compute rows/cols for ~560 repos
n = len(sorted_repos)
cols = 28  # ~20 repos per row
rows = (n + cols - 1) // cols

# Build grid arrays
z = np.full((rows, cols), np.nan)
hover_text = np.full((rows, cols), '', dtype=object)

for idx, (_, row) in enumerate(sorted_repos.iterrows()):
    r = idx // cols
    c = idx % cols
    z[r, c] = row['level']
    hover_text[r, c] = f"{row['repo']}<br>L{row['level']} {row['label']}<br>{row['artifacts']} artifacts<br>{row['confidence']:.0%} confidence"

# Custom colorscale mapping 1-4 to level colors
colorscale = [
    [0.0, '#6b7280'],   # L1 gray
    [0.33, '#3b82f6'],  # L2 blue
    [0.66, '#f97316'],  # L3 orange
    [1.0, '#22c55e'],   # L4 green
]

fig_heatmap = go.Figure(go.Heatmap(
    z=z,
    zmin=1, zmax=4,
    colorscale=colorscale,
    hovertext=hover_text,
    hoverinfo='text',
    showscale=False,
    xgap=2, ygap=2,
))

# Add level legend as annotations
legend_y = -0.08
for lvl, label in LEVEL_LABELS.items():
    count = level_dist.get(lvl, 0)
    fig_heatmap.add_annotation(
        x=(lvl - 1) * 7, y=legend_y, yref='paper',
        text=f'<b style="color:{LEVEL_COLORS[lvl]}">■</b> {label}: {count}',
        showarrow=False, font=dict(size=11),
    )

fig_heatmap.update_layout(
    title=f'Repository Maturity Heatmap — {ORG_NAME} ({n} repos)<br>'
          f'<span style="font-size:12px;color:gray">Each square = 1 repo, sorted by maturity level (highest first)</span>',
    height=max(400, rows * 18 + 120),
    width=1100,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False, autorange='reversed'),
    margin=dict(t=80, b=60, l=10, r=10),
    plot_bgcolor='white',
)
fig_heatmap.show()

# Summary
for lvl in [4, 3, 2, 1]:
    count = level_dist.get(lvl, 0)
    pct = count / total_repos * 100
    print(f"  {LEVEL_LABELS[lvl]:25s} {count:4d} ({pct:5.1f}%)")

## Chart 2: Temporal Maturity Evolution

In [ ]:
# Build monthly timeline of maturity level distribution
# ALL repos counted at every month (constant denominator = total_repos)
# - L1 repos: L1 at every month (never adopted)
# - Genuine repos with level_reached dates: transition when artifacts first appear
# - Genuine repos without dates: assigned current level from today only

all_dates = []
for dates in level_reached.values():
    all_dates.extend(dates.values())

fig_temporal = None
if all_dates:
    min_date = pd.Timestamp(min(all_dates)).tz_localize(None)
    max_date = pd.Timestamp.now()
    months = pd.date_range(start=min_date.to_period('M').to_timestamp(),
                           end=max_date, freq='MS')
    last_month = months[-1]

    timeline_rows = []
    for month in months:
        month_tz = pd.Timestamp(month, tz='UTC')
        counts = {1: 0, 2: 0, 3: 0, 4: 0}

        for repo, score in repo_scores.items():
            if score.overall_level == 1:
                # L1 repos (including overridden) — always L1
                counts[1] += 1
            elif repo in level_reached:
                # Genuine repo with temporal data — use transition dates
                repo_level = 1
                for lvl in [2, 3, 4]:
                    if lvl in level_reached[repo] and level_reached[repo][lvl] <= month_tz:
                        repo_level = lvl
                counts[repo_level] += 1
            else:
                # Genuine repo without temporal data — show at current level only at last month
                if month == last_month:
                    counts[score.overall_level] += 1
                else:
                    counts[1] += 1  # assume L1 before we have data

        row = {'month': month}
        for lvl in [1, 2, 3, 4]:
            row[f'L{lvl}_count'] = counts[lvl]
            row[f'L{lvl}_pct'] = counts[lvl] / total_repos * 100
        row['total'] = total_repos
        timeline_rows.append(row)

    timeline_df = pd.DataFrame(timeline_rows)

    fig_temporal = go.Figure()
    for lvl in [4, 3, 2, 1]:  # Stack from top level down
        fig_temporal.add_trace(go.Scatter(
            x=timeline_df['month'],
            y=timeline_df[f'L{lvl}_pct'],
            name=LEVEL_LABELS[lvl],
            mode='lines',
            stackgroup='one',
            fillcolor=LEVEL_COLORS[lvl],
            line=dict(width=0.5, color=LEVEL_COLORS[lvl]),
            hovertemplate=f'{LEVEL_LABELS[lvl]}: ' + '%{y:.1f}% (%{customdata} repos)<extra></extra>',
            customdata=timeline_df[f'L{lvl}_count'],
        ))

    fig_temporal.update_layout(
        title=f'Maturity Evolution Over Time — {ORG_NAME} ({total_repos} repos)',
        xaxis_title='Month',
        yaxis_title='% of all repositories',
        yaxis=dict(range=[0, 100]),
        height=450,
        width=1000,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        hovermode='x unified',
    )
    fig_temporal.show()

    # Show final month stats
    final = timeline_df.iloc[-1]
    print(f"Final month ({final['month'].strftime('%Y-%m')}):")
    for lvl in [1, 2, 3, 4]:
        print(f"  {LEVEL_LABELS[lvl]}: {int(final[f'L{lvl}_count'])} ({final[f'L{lvl}_pct']:.1f}%)")
else:
    print("No temporal data available — skipping evolution chart.")

In [ ]:
# Build per-repo monthly maturity scores with full fields
# Recomputes maturity score at each month based on which artifacts existed by then
monthly_repo_df = None

# Configurable date range
monthly_start_date = "2024-12-01"
monthly_end_date = "2025-11-30"

if all_dates:
    months = pd.date_range(
        start=monthly_start_date,
        end=monthly_end_date,
        freq='MS',
    )
    last_month = months[-1]

    monthly_repo_rows = []
    for month in months:
        month_tz = pd.Timestamp(month, tz='UTC')
        for repo, score in repo_scores.items():
            # Determine level at this month
            if score.overall_level == 1:
                repo_level = 1
            elif repo in level_reached:
                repo_level = 1
                for lvl in [2, 3, 4]:
                    if lvl in level_reached[repo] and level_reached[repo][lvl] <= month_tz:
                        repo_level = lvl
            else:
                if month == last_month:
                    repo_level = score.overall_level
                else:
                    repo_level = 1

            # Count artifacts at each level that existed by this month
            l2_primary = 0
            l3_primary = 0
            l4_primary = 0
            artifact_count = 0
            tools_at_month = set()

            fc_df = score.file_classifications
            if fc_df is not None and not fc_df.empty and repo in level_reached:
                ts_df = load_timeseries(str(OUTPUT_DIR / repo), repo)
                if ts_df is not None and not ts_df.empty:
                    # Artifacts that existed by this month
                    created = ts_df[
                        (ts_df['action'] == 'created') &
                        (ts_df['commit_date'] <= month_tz)
                    ]['artifact_path'].unique()
                    deleted = ts_df[
                        (ts_df['action'] == 'deleted') &
                        (ts_df['commit_date'] <= month_tz)
                    ]['artifact_path'].unique()
                    alive = set(created) - set(deleted)

                    fc_alive = fc_df[fc_df['artifact_path'].isin(alive)]
                    artifact_count = len(fc_alive)
                    l2_primary = int((fc_alive['assigned_maturity_level'] == 2).sum())
                    l3_primary = int((fc_alive['assigned_maturity_level'] == 3).sum())
                    l4_primary = int((fc_alive['assigned_maturity_level'] == 4).sum())
                    if 'tool_name' in fc_alive.columns:
                        tools_at_month = set(
                            fc_alive.loc[fc_alive['tool_name'] != 'unknown', 'tool_name']
                        )

            # Fallback: use current totals for repos without temporal data
            if artifact_count == 0 and repo_level > 1:
                if month == last_month or score.overall_level == 1:
                    artifact_count = score.artifact_count
                    l2_primary = score.level_evidence.get(2, {}).get('primary', 0)
                    l3_primary = score.level_evidence.get(3, {}).get('primary', 0)
                    l4_primary = score.level_evidence.get(4, {}).get('primary', 0)
                    tools_at_month = set(score.tools_detected)

            monthly_repo_rows.append({
                'month': month.strftime('%Y-%m'),
                'repo': repo,
                'level': repo_level,
                'label': LEVEL_LABELS[repo_level],
                'confidence': round(score.confidence, 3) if repo_level == score.overall_level else (1.0 if repo_level == 1 else round(score.confidence, 3)),
                'artifacts': artifact_count,
                'tools': ', '.join(sorted(tools_at_month)),
                'l2_primary': l2_primary,
                'l3_primary': l3_primary,
                'l4_primary': l4_primary,
            })
    monthly_repo_df = pd.DataFrame(monthly_repo_rows)
    print(f"Monthly scores: {len(monthly_repo_df)} rows "
          f"({len(months)} months x {len(repo_scores)} repos)")
    print(f"Period: {monthly_start_date} to {monthly_end_date}")
else:
    print("No temporal data — monthly scores not available.")

## Chart 3: Most Advanced Repositories

In [ ]:
# Top repos by maturity level, then confidence
top_repos_df = batch_df.sort_values(
    ['level', 'confidence', 'artifacts'],
    ascending=[False, False, False],
).head(30).copy()

top_repos_df['display'] = top_repos_df.apply(
    lambda r: f"{r['repo']} (L{r['level']})", axis=1
)

fig_top_repos = go.Figure(go.Bar(
    y=top_repos_df['display'].tolist()[::-1],
    x=top_repos_df['confidence'].tolist()[::-1],
    orientation='h',
    marker_color=[LEVEL_COLORS[lvl] for lvl in top_repos_df['level'].tolist()[::-1]],
    text=[f"{c:.0%}" for c in top_repos_df['confidence'].tolist()[::-1]],
    textposition='auto',
    hovertemplate='<b>%{y}</b><br>Confidence: %{x:.1%}<br>Artifacts: %{customdata}<extra></extra>',
    customdata=top_repos_df['artifacts'].tolist()[::-1],
))

fig_top_repos.update_layout(
    title=f'Top 30 Repositories by Maturity — {ORG_NAME}',
    xaxis_title='Confidence',
    xaxis=dict(tickformat='.0%', range=[0, 1]),
    height=max(500, len(top_repos_df) * 22 + 100),
    width=900,
    margin=dict(l=250),
)
fig_top_repos.show()

print("\nTop 30 repos:")
display(top_repos_df[['repo', 'level', 'label', 'confidence', 'artifacts', 'tools']].reset_index(drop=True))

## Chart 4: Top AI Artifact Contributors

In [ ]:
# Concatenate timeseries CSVs from genuine L2+ repos only
# and aggregate by author_name_hash
all_ts = []
for repo in tqdm(genuine_repos, desc="Loading timeseries"):
    ts_path = list((OUTPUT_DIR / repo).glob('*_artifact_timeseries.csv'))
    if ts_path:
        df = pd.read_csv(ts_path[0])
        df['repo_name'] = repo
        all_ts.append(df)

fig_contributors = None
contributors_df = pd.DataFrame()
if all_ts:
    combined_ts = pd.concat(all_ts, ignore_index=True)
    print(f"Total timeseries rows: {len(combined_ts)} across {len(all_ts)} genuine repos")

    contributors_df = combined_ts.groupby('author_name_hash').agg(
        total_commits=('commit_sha', 'count'),
        repos_count=('repo_name', 'nunique'),
        repo_list=('repo_name', lambda x: ', '.join(sorted(x.unique()))),
    ).sort_values('total_commits', ascending=False).reset_index()

    print(f"Unique contributors: {len(contributors_df)}")

    # Top 30 contributors bar chart
    top_n = 30
    top_contrib = contributors_df.head(top_n)

    # Shorten hashes for display
    short_labels = [h[:12] + '...' if len(h) > 12 else h for h in top_contrib['author_name_hash']]

    fig_contributors = go.Figure(go.Bar(
        y=short_labels[::-1],
        x=top_contrib['total_commits'].tolist()[::-1],
        orientation='h',
        marker_color='#3b82f6',
        text=top_contrib['total_commits'].tolist()[::-1],
        textposition='auto',
        hovertemplate='<b>%{customdata[0]}</b><br>Commits: %{x}<br>Repos: %{customdata[1]}<extra></extra>',
        customdata=list(zip(
            top_contrib['author_name_hash'].tolist()[::-1],
            top_contrib['repos_count'].tolist()[::-1],
        )),
    ))

    fig_contributors.update_layout(
        title=f'Top {top_n} AI Artifact Contributors — {ORG_NAME} (anonymized)',
        xaxis_title='Total artifact commits',
        height=max(500, top_n * 22 + 100),
        width=900,
        margin=dict(l=150),
    )
    fig_contributors.show()

    print(f"\nTop 10 contributors:")
    display(top_contrib[['author_name_hash', 'total_commits', 'repos_count']].head(10))
else:
    print("No timeseries data available.")

## Chart 5: Tool Adoption Breakdown

In [ ]:
# Count tool usage across genuine L2+ repos only
tool_repo_counts = {}   # tool -> set of repos
tool_artifact_counts = {}  # tool -> total artifact count

for repo in genuine_repos:
    score = repo_scores[repo]
    for tool in score.tools_detected:
        tool_repo_counts.setdefault(tool, set()).add(repo)
    # Also count from file classifications
    fc_df = score.file_classifications
    if fc_df is not None and not fc_df.empty:
        for tool, group in fc_df[fc_df['tool_name'] != 'unknown'].groupby('tool_name'):
            tool_artifact_counts[tool] = tool_artifact_counts.get(tool, 0) + len(group)

tool_adoption_df = pd.DataFrame([
    {
        'tool': tool,
        'repos': len(repos_set),
        'pct_repos': len(repos_set) / len(genuine_repos) * 100 if genuine_repos else 0,
        'artifacts': tool_artifact_counts.get(tool, 0),
    }
    for tool, repos_set in tool_repo_counts.items()
]).sort_values('repos', ascending=False).reset_index(drop=True)

fig_tools = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Repos Using Each Tool', 'Artifacts per Tool'],
    column_widths=[0.5, 0.5],
)

# Bar: repo count per tool
fig_tools.add_trace(go.Bar(
    y=tool_adoption_df['tool'].tolist()[::-1],
    x=tool_adoption_df['repos'].tolist()[::-1],
    orientation='h',
    marker_color='#3b82f6',
    text=[f"{r} ({p:.0f}%)" for r, p in zip(
        tool_adoption_df['repos'].tolist()[::-1],
        tool_adoption_df['pct_repos'].tolist()[::-1],
    )],
    textposition='auto',
    showlegend=False,
), row=1, col=1)

# Bar: artifact count per tool
fig_tools.add_trace(go.Bar(
    y=tool_adoption_df['tool'].tolist()[::-1],
    x=tool_adoption_df['artifacts'].tolist()[::-1],
    orientation='h',
    marker_color='#f97316',
    text=tool_adoption_df['artifacts'].tolist()[::-1],
    textposition='auto',
    showlegend=False,
), row=1, col=2)

fig_tools.update_layout(
    title=f'Tool Adoption — {ORG_NAME} ({len(genuine_repos)} repos with AI artifacts)',
    height=max(400, len(tool_adoption_df) * 30 + 100),
    width=1000,
)
fig_tools.show()

display(tool_adoption_df)

## Chart 6: Category Gap Analysis

In [ ]:
# For each of 9 categories, count how many genuine L2+ repos have >= 1 artifact
category_repo_counts = {cat: 0 for cat in CATEGORY_NAMES}
category_artifact_totals = {cat: 0 for cat in CATEGORY_NAMES}

for repo in genuine_repos:
    score = repo_scores[repo]
    for cat, count in score.category_counts.items():
        if count > 0:
            category_repo_counts[cat] = category_repo_counts.get(cat, 0) + 1
            category_artifact_totals[cat] = category_artifact_totals.get(cat, 0) + count

cat_coverage_df = pd.DataFrame([
    {
        'category': cat,
        'level': f"L{CATEGORY_TO_LEVEL[cat]}",
        'level_num': int(CATEGORY_TO_LEVEL[cat]),
        'repos_with': category_repo_counts.get(cat, 0),
        'pct_of_scored': category_repo_counts.get(cat, 0) / len(genuine_repos) * 100 if genuine_repos else 0,
        'total_artifacts': category_artifact_totals.get(cat, 0),
    }
    for cat in CATEGORY_NAMES
]).sort_values('repos_with', ascending=True)

fig_categories = go.Figure(go.Bar(
    y=[f"{r['category']} ({r['level']})" for _, r in cat_coverage_df.iterrows()],
    x=cat_coverage_df['pct_of_scored'].tolist(),
    orientation='h',
    marker_color=[LEVEL_COLORS[r['level_num']] for _, r in cat_coverage_df.iterrows()],
    text=[f"{p:.0f}% ({n} repos)" for p, n in zip(
        cat_coverage_df['pct_of_scored'], cat_coverage_df['repos_with']
    )],
    textposition='auto',
))

fig_categories.update_layout(
    title=f'Category Coverage — {ORG_NAME} ({len(genuine_repos)} repos with AI artifacts)',
    xaxis_title='% of L2+ repos with this category',
    xaxis=dict(range=[0, 105]),
    height=450,
    width=900,
    margin=dict(l=200),
)
fig_categories.show()

display(cat_coverage_df[['category', 'level', 'repos_with', 'pct_of_scored', 'total_artifacts']].sort_values('pct_of_scored', ascending=False))

## Chart 7: Artifact Density Distribution

In [ ]:
# Distribution of artifact counts per repo
density_df = batch_df[['repo', 'artifacts', 'level']].copy()

fig_density = go.Figure()

for lvl in [1, 2, 3, 4]:
    lvl_data = density_df[density_df['level'] == lvl]
    if not lvl_data.empty:
        fig_density.add_trace(go.Histogram(
            x=lvl_data['artifacts'],
            name=LEVEL_LABELS[lvl],
            marker_color=LEVEL_COLORS[lvl],
            opacity=0.7,
        ))

fig_density.update_layout(
    title=f'Artifact Count Distribution — {ORG_NAME}',
    xaxis_title='Artifacts per repository',
    yaxis_title='Repository count',
    barmode='stack',
    height=400,
    width=900,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig_density.show()

print(f"Artifact count statistics:")
print(f"  Mean: {density_df['artifacts'].mean():.1f}")
print(f"  Median: {density_df['artifacts'].median():.0f}")
print(f"  Max: {density_df['artifacts'].max()} ({density_df.loc[density_df['artifacts'].idxmax(), 'repo']})")
print(f"  Repos with 0 artifacts: {(density_df['artifacts'] == 0).sum()}")

## Chart 8: Risk Flags

In [ ]:
# Collect risk flags from genuine L2+ repos only
risk_rows = []

for repo in genuine_repos:
    score = repo_scores[repo]

    # Red coherence flags
    for flag in score.coherence_flags:
        if flag.status == 'red':
            risk_rows.append({
                'repo': repo,
                'level': score.overall_level,
                'flag_type': 'Coherence',
                'severity': 'high',
                'message': f"{flag.check}: {flag.message}",
            })

    # Low confidence at high levels
    if score.overall_level >= 3 and score.confidence < 0.5:
        risk_rows.append({
            'repo': repo,
            'level': score.overall_level,
            'flag_type': 'Low confidence',
            'severity': 'medium',
            'message': f"L{score.overall_level} with only {score.confidence:.0%} confidence",
        })

    # Temporal health concerns
    if repo in repo_temporal:
        th = repo_temporal[repo]
        for v in th.health_verdicts:
            if v['verdict'] == 'concern':
                risk_rows.append({
                    'repo': repo,
                    'level': score.overall_level,
                    'flag_type': 'Temporal health',
                    'severity': 'high',
                    'message': v['message'],
                })
            elif v['verdict'] == 'warning':
                risk_rows.append({
                    'repo': repo,
                    'level': score.overall_level,
                    'flag_type': 'Temporal health',
                    'severity': 'medium',
                    'message': v['message'],
                })

        # Bus-factor risk: single-author grounding categories
        for cat, author_count in th.author_diversity.items():
            if author_count <= 1 and cat in GROUNDING_CATEGORIES:
                risk_rows.append({
                    'repo': repo,
                    'level': score.overall_level,
                    'flag_type': 'Bus factor',
                    'severity': 'medium',
                    'message': f"Single author for {cat} artifacts",
                })

risk_df = pd.DataFrame(risk_rows)

if not risk_df.empty:
    # Summary by flag type
    risk_summary = risk_df.groupby(['flag_type', 'severity']).size().reset_index(name='count')

    severity_colors = {'high': '#ef4444', 'medium': '#eab308', 'low': '#6b7280'}

    fig_risk = go.Figure(go.Bar(
        y=risk_summary['flag_type'],
        x=risk_summary['count'],
        orientation='h',
        marker_color=[severity_colors.get(s, '#6b7280') for s in risk_summary['severity']],
        text=[f"{c} ({s})" for c, s in zip(risk_summary['count'], risk_summary['severity'])],
        textposition='auto',
    ))

    fig_risk.update_layout(
        title=f'Risk Flags Summary — {ORG_NAME}',
        xaxis_title='Number of flags',
        height=300,
        width=700,
    )
    fig_risk.show()

    print(f"\nTotal risk flags: {len(risk_df)}")
    print(f"Repos affected: {risk_df['repo'].nunique()}")
    print(f"\nHigh severity flags (showing first 20):")
    display(risk_df[risk_df['severity'] == 'high'][['repo', 'level', 'flag_type', 'message']].head(20))
else:
    fig_risk = None
    print("No risk flags found.")

## Phase 3: LLM Executive Summary (Optional)

Requires `ANTHROPIC_API_KEY` environment variable.

In [ ]:
llm_narrative = None

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY not set — skipping LLM narrative.")
else:
    import anthropic
    client = anthropic.Anthropic()

    # Build context for the LLM
    level_dist_text = "\n".join(
        f"  L{lvl}: {level_dist.get(lvl, 0)} repos ({level_dist.get(lvl, 0)/total_repos*100:.1f}%)"
        for lvl in [1, 2, 3, 4]
    )

    tool_text = tool_adoption_df.to_string(index=False) if not tool_adoption_df.empty else "No tools detected"

    cat_text = cat_coverage_df.sort_values('pct_of_scored', ascending=False).to_string(index=False)

    top_repos_text = top_repos_df[['repo', 'level', 'confidence', 'artifacts', 'tools']].head(10).to_string(index=False)

    risk_text = ""
    if not risk_df.empty:
        risk_summary_text = risk_df.groupby('flag_type').size().to_string()
        risk_text = f"Risk flags: {len(risk_df)} total across {risk_df['repo'].nunique()} repos\n{risk_summary_text}"

    contributor_text = ""
    if not contributors_df.empty:
        contributor_text = f"Unique contributors: {len(contributors_df)}\nTop contributor: {contributors_df.iloc[0]['total_commits']} commits across {contributors_df.iloc[0]['repos_count']} repos"

    prompt = f"""You are an AI adoption analyst for an anonymous benchmark study.

Generate an EXECUTIVE SUMMARY for the VP of Engineering about AI tool adoption maturity across the "{ORG_NAME}" organization.

## Organization Data

**Total repositories**: {total_repos}
**Repos with AI artifacts (L2+)**: {scored_with_artifacts} ({scored_with_artifacts/total_repos*100:.0f}%)

**Maturity Level Distribution**:
{level_dist_text}

**Tool Adoption**:
{tool_text}

**Category Coverage** (across repos with AI artifacts):
{cat_text}

**Top 10 Most Mature Repositories**:
{top_repos_text}

**Contributors**:
{contributor_text}

**Risk Flags**:
{risk_text}

## Maturity Levels Reference
- L1 Ad Hoc: No AI-specific artifacts
- L2 Grounded Prompting: Rules, configuration, architecture, code-style files
- L3 Agent-Augmented: Agent definitions, commands, skills
- L4 Agentic Orchestration: Multi-agent flows, session logs

## Instructions

Write a structured report with:
1. **Executive Summary** (3-4 sentences: overall adoption state, key metric, trajectory)
2. **Adoption Landscape** — Distribution analysis, what the levels mean in practice
3. **Tool Strategy** — Which tools dominate, fragmentation vs standardization
4. **Maturity Gaps** — Which categories are underrepresented and what that means
5. **Risk Assessment** — Temporal health concerns, bus-factor risks
6. **Recommendations** — Top 3-5 actionable steps for the VP to drive adoption

Be specific with numbers. Use markdown formatting. There are exactly 4 maturity levels (L1-L4), no L5.
"""

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=3000,
        messages=[{"role": "user", "content": prompt}],
    )

    llm_narrative = response.content[0].text
    print(llm_narrative)

## Phase 4: PDF Export

In [ ]:
# ── Org-level PDF report template ──

ORG_REPORT_CSS = """
@page {
    size: A4;
    margin: 2cm 1.5cm;
    @bottom-center {
        content: "Page " counter(page) " of " counter(pages);
        font-size: 9pt;
        color: #6b7280;
    }
}

body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif;
    font-size: 11pt;
    line-height: 1.5;
    color: #1f2937;
    max-width: 100%;
}

h1 {
    font-size: 22pt;
    color: #111827;
    margin-top: 0;
    border-bottom: 3px solid #3b82f6;
    padding-bottom: 8px;
}

h2 {
    font-size: 16pt;
    color: #1f2937;
    margin-top: 24px;
    border-bottom: 1px solid #e5e7eb;
    padding-bottom: 4px;
}

h3 {
    font-size: 13pt;
    color: #374151;
    margin-top: 16px;
}

.title-page {
    text-align: center;
    padding-top: 100px;
    page-break-after: always;
}

.title-page h1 {
    font-size: 28pt;
    border-bottom: none;
    margin-bottom: 8px;
}

.title-page .subtitle {
    font-size: 14pt;
    color: #6b7280;
    margin-bottom: 40px;
}

.metric-box {
    display: inline-block;
    background: #f9fafb;
    border: 1px solid #e5e7eb;
    border-radius: 8px;
    padding: 12px 20px;
    margin: 6px 8px 6px 0;
    text-align: center;
}

.metric-value {
    font-size: 20pt;
    font-weight: bold;
    color: #111827;
}

.metric-label {
    font-size: 9pt;
    color: #6b7280;
}

.page-break { page-break-before: always; }

table {
    width: 100%;
    border-collapse: collapse;
    margin: 12px 0;
    font-size: 10pt;
}

th {
    background: #f9fafb;
    border: 1px solid #e5e7eb;
    padding: 8px 10px;
    text-align: left;
    font-weight: 600;
}

td {
    border: 1px solid #e5e7eb;
    padding: 6px 10px;
}

tr:nth-child(even) td {
    background: #f9fafb;
}

.chart-container {
    margin: 16px 0;
    text-align: center;
}

.chart-container img {
    max-width: 100%;
    height: auto;
}

.chart-description {
    font-size: 10pt;
    color: #6b7280;
    font-style: italic;
    margin: 4px 0 16px 0;
}

.badge {
    display: inline-block;
    font-size: 14pt;
    font-weight: bold;
    padding: 6px 16px;
    border-radius: 8px;
    margin: 4px 4px;
}

.badge-l1 { background: #f3f4f6; color: #6b7280; }
.badge-l2 { background: #dbeafe; color: #1d4ed8; }
.badge-l3 { background: #ffedd5; color: #c2410c; }
.badge-l4 { background: #dcfce7; color: #15803d; }

.level-bar {
    display: flex;
    gap: 0;
    height: 40px;
    border-radius: 6px;
    overflow: hidden;
    margin: 16px 0;
}

.level-bar div {
    display: flex;
    align-items: center;
    justify-content: center;
    color: white;
    font-weight: bold;
    font-size: 10pt;
}

.llm-report {
    background: #f9fafb;
    border: 1px solid #e5e7eb;
    padding: 14px 18px;
    font-size: 10pt;
    page-break-inside: auto;
    box-decoration-break: clone;
    -webkit-box-decoration-break: clone;
}

.risk-high { color: #dc2626; font-weight: bold; }
.risk-medium { color: #d97706; }

.about-section p {
    margin: 6px 0;
    font-size: 10pt;
}

.disclaimer {
    background: #fefce8;
    border: 1px solid #fde68a;
    border-radius: 6px;
    padding: 10px 14px;
    font-size: 9pt;
    color: #92400e;
}
"""

print("CSS defined.")

In [ ]:
from jinja2 import Template

ORG_REPORT_TEMPLATE = Template("""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<style>{{ css }}</style>
</head>
<body>

<!-- Page 1: Title -->
<div class="title-page">
    <h1>AI Adoption Maturity Report</h1>
    <div class="subtitle">Organization: {{ org_name }}</div>

    <div class="level-bar">
        {% for lvl in [1, 2, 3, 4] %}
        {% set pct = level_pcts[lvl] %}
        {% if pct > 0 %}
        <div style="width: {{ pct }}%; background: {{ level_colors[lvl] }};">
            L{{ lvl }}: {{ pct | round(0) | int }}%
        </div>
        {% endif %}
        {% endfor %}
    </div>

    <div style="margin-top: 30px;">
        <div class="metric-box">
            <div class="metric-value">{{ total_repos }}</div>
            <div class="metric-label">Total Repositories</div>
        </div>
        <div class="metric-box">
            <div class="metric-value">{{ scored_with_artifacts }}</div>
            <div class="metric-label">With AI Artifacts</div>
        </div>
        <div class="metric-box">
            <div class="metric-value">{{ adoption_pct }}%</div>
            <div class="metric-label">Adoption Rate</div>
        </div>
        <div class="metric-box">
            <div class="metric-value">{{ unique_contributors }}</div>
            <div class="metric-label">Contributors</div>
        </div>
    </div>

    <p style="margin-top: 40px; color: #6b7280; font-size: 10pt;">Generated: {{ date }}</p>
    <p style="color: #6b7280; font-size: 9pt;">AIME Framework</p>
</div>

<!-- Page 2: About -->
<div class="page-break about-section">
    <h1>About This Report</h1>

    <h2>Methodology</h2>
    <p>This report uses the <strong>AIME (AI Adoption Maturity Evaluator)</strong> framework
    to assess AI tool adoption across all repositories in the organization.</p>
    <p>Each repository is scored using three signals:</p>
    <ul>
        <li><strong>Tool Detection</strong> &mdash; Pattern matching against 14+ known AI tool configurations</li>
        <li><strong>Path Semantic Intent</strong> &mdash; Embedding-based classification of artifact file paths</li>
        <li><strong>Content Semantic Classification</strong> &mdash; Embedding-based classification of artifact content</li>
    </ul>

    <h2>Maturity Levels</h2>
    <table>
        <tr><th>Level</th><th>Name</th><th>Description</th></tr>
        <tr><td><span class="badge badge-l1">L1</span></td><td>Ad Hoc</td><td>No AI-specific configuration artifacts found</td></tr>
        <tr><td><span class="badge badge-l2">L2</span></td><td>Grounded Prompting</td><td>Rules, configuration, architecture, code-style files</td></tr>
        <tr><td><span class="badge badge-l3">L3</span></td><td>Agent-Augmented</td><td>Agent definitions, commands, skills</td></tr>
        <tr><td><span class="badge badge-l4">L4</span></td><td>Agentic Orchestration</td><td>Multi-agent flows, session logs</td></tr>
    </table>

    <div class="disclaimer">
        <strong>Note:</strong> Maturity scores reflect the presence and nature of AI tool configuration
        artifacts in version control. They do not measure code quality, developer productivity,
        or the effectiveness of AI tool usage. Author identities are anonymized via HMAC hashing.
    </div>
</div>

<!-- Page 3: Maturity Heatmap -->
<div class="page-break">
    <h1>Maturity Heatmap</h1>
    <p>Each tile represents one repository, colored by its maturity level.
    Tiles are grouped by level, providing an at-a-glance view of the entire organization.</p>
    {% if figures.heatmap %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.heatmap }}">
    </div>
    {% endif %}

    <h3>Level Distribution</h3>
    <table>
        <tr><th>Level</th><th>Count</th><th>%</th></tr>
        {% for lvl in [1, 2, 3, 4] %}
        <tr>
            <td>{{ level_labels[lvl] }}</td>
            <td>{{ level_counts[lvl] }}</td>
            <td>{{ "%.1f" | format(level_pcts[lvl]) }}%</td>
        </tr>
        {% endfor %}
    </table>
</div>

<!-- Page 4: Temporal Evolution -->
<div class="page-break">
    <h1>Maturity Evolution Over Time</h1>
    <p>Shows how the distribution of maturity levels has changed over time.
    Based on git commit history of AI artifacts across {{ repos_with_temporal }} repositories with temporal data.</p>
    {% if figures.temporal %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.temporal }}">
    </div>
    {% else %}
    <p><em>No temporal data available.</em></p>
    {% endif %}
</div>

<!-- Page 5: Top Repos + Tool Adoption -->
<div class="page-break">
    <h1>Top Repositories</h1>
    <p>Repositories ranked by maturity level and confidence score.</p>
    {% if figures.top_repos %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.top_repos }}">
    </div>
    {% endif %}

    <h2>Tool Adoption</h2>
    <p>AI tools detected across the organization.</p>
    {% if figures.tools %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.tools }}">
    </div>
    {% endif %}
</div>

<!-- Page 6: Category Gaps + Density -->
<div class="page-break">
    <h1>Category Gap Analysis</h1>
    <p>Coverage of each maturity category across repositories with AI artifacts.
    Low coverage suggests an area where adoption could be expanded.</p>
    {% if figures.categories %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.categories }}">
    </div>
    {% endif %}

    <h2>Artifact Density</h2>
    <p>Distribution of artifact counts across repositories.</p>
    {% if figures.density %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.density }}">
    </div>
    {% endif %}
</div>

<!-- Page 7: Contributors + Risk -->
<div class="page-break">
    <h1>Top Contributors</h1>
    <p>Most active contributors to AI tool configuration artifacts (anonymized).</p>
    {% if figures.contributors %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.contributors }}">
    </div>
    {% endif %}

    <h2>Risk Flags</h2>
    {% if risk_count > 0 %}
    <p>{{ risk_count }} flags across {{ risk_repos }} repositories.</p>
    {% if figures.risk %}
    <div class="chart-container">
        <img src="data:image/png;base64,{{ figures.risk }}">
    </div>
    {% endif %}

    <table>
        <tr><th>Repository</th><th>Level</th><th>Type</th><th>Severity</th><th>Details</th></tr>
        {% for flag in risk_flags_top %}
        <tr>
            <td>{{ flag.repo }}</td>
            <td>L{{ flag.level }}</td>
            <td>{{ flag.flag_type }}</td>
            <td class="risk-{{ flag.severity }}">{{ flag.severity }}</td>
            <td>{{ flag.message }}</td>
        </tr>
        {% endfor %}
    </table>
    {% else %}
    <p>No risk flags identified.</p>
    {% endif %}
</div>

<!-- Page 8+: LLM Narrative (optional) -->
{% if llm_html %}
<div class="page-break">
    <h1>AI-Generated Executive Analysis</h1>
    <div class="llm-report">
        {{ llm_html }}
    </div>
    <p class="chart-description">Generated by Claude (Sonnet). Based on quantitative assessment data only.</p>
</div>
{% endif %}

</body>
</html>
""")

print("HTML template defined.")

In [ ]:
# Convert figures to base64 and render PDF
pdf_figures = {}
chart_configs = {
    'heatmap': (fig_heatmap, 1100, 700),
    'temporal': (fig_temporal, 1000, 450),
    'top_repos': (fig_top_repos, 900, max(500, len(top_repos_df) * 22 + 100)),
    'tools': (fig_tools, 1000, max(400, len(tool_adoption_df) * 30 + 100)),
    'categories': (fig_categories, 900, 450),
    'density': (fig_density, 900, 400),
    'contributors': (fig_contributors, 900, 700) if fig_contributors else (None, 0, 0),
    'risk': (fig_risk, 700, 300) if 'fig_risk' in dir() and fig_risk is not None else (None, 0, 0),
}

for name, (fig, w, h) in chart_configs.items():
    if fig is not None:
        pdf_figures[name] = fig_to_base64(fig, width=w, height=h)
        print(f"  Encoded: {name} ({w}x{h})")

# Build template context
level_counts = {lvl: int(level_dist.get(lvl, 0)) for lvl in [1, 2, 3, 4]}
level_pcts = {lvl: level_counts[lvl] / total_repos * 100 for lvl in [1, 2, 3, 4]}

# Risk flags for table (top 30)
risk_flags_top = []
if not risk_df.empty:
    for _, row in risk_df.sort_values('severity').head(30).iterrows():
        risk_flags_top.append(row)

html_content = ORG_REPORT_TEMPLATE.render(
    css=ORG_REPORT_CSS,
    org_name=ORG_NAME,
    date=datetime.now().strftime("%Y-%m-%d"),
    total_repos=total_repos,
    scored_with_artifacts=scored_with_artifacts,
    adoption_pct=round(scored_with_artifacts / total_repos * 100),
    unique_contributors=len(contributors_df) if not contributors_df.empty else 0,
    level_colors=LEVEL_COLORS,
    level_labels=LEVEL_LABELS,
    level_counts=level_counts,
    level_pcts=level_pcts,
    figures=pdf_figures,
    repos_with_temporal=len(level_reached),
    risk_count=len(risk_df) if not risk_df.empty else 0,
    risk_repos=risk_df['repo'].nunique() if not risk_df.empty else 0,
    risk_flags_top=risk_flags_top,
    llm_html=_markdown_to_html(llm_narrative) if llm_narrative else None,
)

# Write HTML to temp file, then call WeasyPrint as subprocess
# (in-process WeasyPrint crashes Jupyter kernels due to pango/gobject library conflicts)
import subprocess, tempfile

pdf_path = str(EXPORT_DIR / f'{ORG_NAME}_org_maturity_report.pdf')
html_path = str(EXPORT_DIR / f'{ORG_NAME}_org_maturity_report.html')

with open(html_path, 'w') as f:
    f.write(html_content)
print(f"HTML written: {html_path}")

# Find brew lib path for pango
try:
    brew_prefix = subprocess.check_output(["brew", "--prefix"], text=True, stderr=subprocess.DEVNULL).strip()
    brew_lib = os.path.join(brew_prefix, "lib")
except Exception:
    brew_lib = "/opt/homebrew/lib"

env = os.environ.copy()
env["DYLD_FALLBACK_LIBRARY_PATH"] = brew_lib

result = subprocess.run(
    [sys.executable, "-c",
     f"from weasyprint import HTML; HTML(filename='{html_path}').write_pdf('{pdf_path}')"],
    env=env, capture_output=True, text=True, timeout=120,
)

if result.returncode == 0:
    print(f"PDF exported: {pdf_path}")
else:
    print(f"WeasyPrint error:\n{result.stderr[-500:]}")

In [ ]:
# Export batch results CSV
batch_csv_path = EXPORT_DIR / f'{ORG_NAME}_org_maturity_scores.csv'
batch_df.to_csv(batch_csv_path, index=False)
print(f"Batch scores: {batch_csv_path}")

# Export monthly maturity scores CSV
if monthly_repo_df is not None and not monthly_repo_df.empty:
    monthly_csv_path = EXPORT_DIR / f'{ORG_NAME}_org_maturity_scores_monthly.csv'
    monthly_repo_df.to_csv(monthly_csv_path, index=False)
    print(f"Monthly scores: {monthly_csv_path}")

# Export contributors CSV
if not contributors_df.empty:
    contrib_csv_path = EXPORT_DIR / f'{ORG_NAME}_org_contributors.csv'
    contributors_df.to_csv(contrib_csv_path, index=False)
    print(f"Contributors: {contrib_csv_path}")

# Export risk flags CSV
if not risk_df.empty:
    risk_csv_path = EXPORT_DIR / f'{ORG_NAME}_org_risk_flags.csv'
    risk_df.to_csv(risk_csv_path, index=False)
    print(f"Risk flags: {risk_csv_path}")

In [ ]:
# Unload model
print("Unloading model...")
del model
gc.collect()
print("Done.")